In [4]:
!pip install torch

In [5]:
import torch 
# Check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Example: move a tensor to the selected device
x = torch.rand(3, 3).to(device)
print(f"Tensor is on: {x.device}")

Using GPU: Tesla T4
Tensor is on: cuda:0


In [17]:
# ===== FINAL FIXED GPU CARO TRAINER (Single Cell) =====

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt

# =============================
# GPU SETUP
# =============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# =============================
# GAME SETTINGS
# =============================

BOARD_SIZE = 6
WIN_LEN = 4

# =============================
# CARO ENVIRONMENT
# =============================

class CaroEnv:

    def __init__(self):
        self.reset()

    def reset(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.int8)
        self.player = 1
        self.moves = 0
        return self.get_state()

    def get_state(self):
        return torch.tensor(self.board, dtype=torch.float32)

    def actions(self):
        return np.argwhere(self.board == 0)

    def step(self, action):

        x, y = action

        if self.board[x, y] != 0:
            return self.get_state(), -1, True

        self.board[x, y] = self.player
        self.moves += 1

        done, reward = self.check(x, y)

        if self.moves >= BOARD_SIZE * BOARD_SIZE:
            done = True
            reward = 0

        self.player *= -1

        return self.get_state(), reward, done

    def line(self, x, y, dx, dy):

        p = self.board[x, y]
        c = 1

        i = 1
        while 0 <= x+i*dx < BOARD_SIZE and 0 <= y+i*dy < BOARD_SIZE and self.board[x+i*dx, y+i*dy] == p:
            c += 1
            i += 1

        i = 1
        while 0 <= x-i*dx < BOARD_SIZE and 0 <= y-i*dy < BOARD_SIZE and self.board[x-i*dx, y-i*dy] == p:
            c += 1
            i += 1

        return c

    def check(self, x, y):

        if (
            self.line(x,y,1,0) >= WIN_LEN or
            self.line(x,y,0,1) >= WIN_LEN or
            self.line(x,y,1,1) >= WIN_LEN or
            self.line(x,y,1,-1) >= WIN_LEN
        ):
            return True, 1

        return False, 0


# =============================
# CNN MODEL
# =============================

class CaroNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * BOARD_SIZE * BOARD_SIZE, 256),
            nn.ReLU(),
            nn.Linear(256, BOARD_SIZE * BOARD_SIZE)
        )

    def forward(self, x):

        x = x.unsqueeze(1)
        x = self.conv(x)
        x = self.head(x)

        return x


# =============================
# TRAINING SETUP
# =============================

model = CaroNet().to(device)
target = CaroNet().to(device)

target.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

gamma = 0.97

epsilon = 1.0
epsilon_decay = 0.9995
epsilon_min = 0.05

memory = deque(maxlen=50000)

# =============================
# REPLAY BUFFER
# =============================

def store(exp):
    memory.append(exp)

def sample(batch_size):

    idx = np.random.choice(len(memory), batch_size, replace=False)
    return [memory[i] for i in idx]


# =============================
# TRAIN STEP
# =============================

def train_step(batch_size=256):

    if len(memory) < batch_size:
        return

    batch = sample(batch_size)

    states = torch.stack([b[0] for b in batch]).to(device)
    actions = torch.tensor([b[1] for b in batch]).to(device)
    rewards = torch.tensor([b[2] for b in batch]).float().to(device)
    next_states = torch.stack([b[3] for b in batch]).to(device)
    dones = torch.tensor([b[4] for b in batch]).float().to(device)

    q = model(states)
    next_q = target(next_states).max(1)[0]

    target_q = q.clone()

    target_q[range(batch_size), actions] = rewards + gamma * next_q * (1 - dones)

    loss = loss_fn(q, target_q)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


# =============================
# TRAINING LOOP
# =============================

ENVS = 8
envs = [CaroEnv() for _ in range(ENVS)]

episodes = 2000
reward_history = []

for ep in range(episodes):

    states = [env.reset() for env in envs]
    dones = [False] * ENVS

    total_reward = 0
    step_count = 0

    while not all(dones):

        step_count += 1

        if step_count % 100 == 0:
            print("Episode", ep, "step", step_count)

        for i, env in enumerate(envs):

            if dones[i]:
                continue

            state = states[i]

            acts = env.actions()

            if len(acts) == 0:
                dones[i] = True
                continue

            if random.random() < epsilon:

                a = random.choice(acts)
                action = a[0] * BOARD_SIZE + a[1]

            else:

                with torch.no_grad():

                    q = model(state.unsqueeze(0).to(device))[0].cpu()

                    mask = torch.full((BOARD_SIZE * BOARD_SIZE,), -1e9)

                    for a in acts:
                        idx = a[0] * BOARD_SIZE + a[1]
                        mask[idx] = q[idx]

                    action = torch.argmax(mask).item()

            x = action // BOARD_SIZE
            y = action % BOARD_SIZE

            next_state, reward, done = env.step((x,y))

            store((state, action, reward, next_state, done))

            states[i] = next_state
            dones[i] = done

            total_reward += reward

        if len(memory) > 2000:
            for _ in range(3):
                train_step()

    reward_history.append(total_reward)

    epsilon = max(epsilon * epsilon_decay, epsilon_min)

    if ep % 50 == 0:
        target.load_state_dict(model.state_dict())

    print("Episode:", ep, "Reward:", total_reward)

print("Training finished")


# =============================
# TRAINING PLOT
# =============================

window = 100
avg = []

for i in range(len(reward_history)):
    avg.append(np.mean(reward_history[max(0, i-window):i+1]))

plt.figure()
plt.plot(avg)
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.title("Training Progress")
plt.show()

Device: cuda
GPU: Tesla T4
Episode: 0 Reward: 8
Episode: 1 Reward: 8
Episode: 2 Reward: 8
Episode: 3 Reward: 8
Episode: 4 Reward: 8
Episode: 5 Reward: 8
Episode: 6 Reward: 8
Episode: 7 Reward: 8
Episode: 8 Reward: 8
Episode: 9 Reward: 8
Episode: 10 Reward: 8
Episode: 11 Reward: 8
Episode: 12 Reward: 8
Episode: 13 Reward: 8
Episode: 14 Reward: 8
Episode: 15 Reward: 6
Episode: 16 Reward: 8
Episode: 17 Reward: 8
Episode: 18 Reward: 7
Episode: 19 Reward: 8
Episode: 20 Reward: 8
Episode: 21 Reward: 8
Episode: 22 Reward: 8
Episode: 23 Reward: 8
Episode: 24 Reward: 8
Episode: 25 Reward: 8
Episode: 26 Reward: 8
Episode: 27 Reward: 8
Episode: 28 Reward: 8
Episode: 29 Reward: 8
Episode: 30 Reward: 7
Episode: 31 Reward: 8
Episode: 32 Reward: 8
Episode: 33 Reward: 8
Episode: 34 Reward: 8
Episode: 35 Reward: 8
Episode: 36 Reward: 8
Episode: 37 Reward: 8
Episode: 38 Reward: 8
Episode: 39 Reward: 8
Episode: 40 Reward: 8
Episode: 41 Reward: 8
Episode: 42 Reward: 8
Episode: 43 Reward: 7
Episode: 44 Rew

KeyboardInterrupt: 